In [ ]:
!pip install pandas numpy rdkit xgboost optuna scikit-learn matplotlib joblib -q

# HOMO Regression Pipeline
**Task 6 — Iteration n | Molecular Energy Level Prediction**

This notebook trains a regression model to predict **HOMO** energy levels from molecular descriptors.

**Datasets used:**
- `xy_df.csv` — HOPV + SQL molecules with pre-computed RDKit features and targets
- `extra_data_200k.csv` — 20,000 additional molecules (features computed here)

**Features:**
- 176 RDKit molecular descriptors
- 2048-bit Morgan fingerprints radius 2 (ECFP4, count-based)
- 2048-bit Morgan fingerprints radius 3 (ECFP6, count-based)
- Total: ~4,272 features

**Output:** `homo_model.joblib` + `homo_features.joblib`

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import rdFingerprintGenerator

from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt

RANDOM_STATE = 42
TARGET = 'homo'
print('All imports OK')

## 1. Load xy_df (HOPV + SQL)

In [ ]:
xy = pd.read_csv('xy_df.csv')

# Feature columns — exclude smiles, both targets and stability
RDKIT_COLS = [c for c in xy.columns if c not in ['smiles', 'homo', 'lumo', 'stability_encoded']]

print(f'xy_df shape: {xy.shape}')
print(f'RDKit descriptor columns: {len(RDKIT_COLS)}')
print(f'Rows with HOMO target: {xy[TARGET].notna().sum()}')

# Keep only rows with HOMO target
xy_homo = xy[xy[TARGET].notna()].reset_index(drop=True)
print(f'Usable rows for HOMO training: {len(xy_homo)}')

## 2. Feature Engineering — Morgan Fingerprints

Morgan fingerprints (count-based ECFP) capture substructural patterns like π-conjugation, donor/acceptor motifs, and ring systems — all directly relevant to HOMO energy levels. We add both radius 2 (ECFP4) and radius 3 (ECFP6) for maximum coverage.

In [ ]:
def compute_rdkit_features(smiles_list, rdkit_cols):
    """Compute RDKit descriptors + Morgan fingerprints for a list of SMILES."""
    calculator = MoleculeDescriptors.MolecularDescriptorCalculator(rdkit_cols)
    gen_r2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    gen_r3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)

    desc_rows, fp_r2_rows, fp_r3_rows = [], [], []
    failed = 0

    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            desc_rows.append([np.nan] * len(rdkit_cols))
            fp_r2_rows.append([0] * 2048)
            fp_r3_rows.append([0] * 2048)
            failed += 1
        else:
            desc_rows.append(list(calculator.CalcDescriptors(mol)))
            # Count-based Morgan fingerprints
            fp_r2 = np.zeros(2048, dtype=np.float32)
            for idx, count in gen_r2.GetCountFingerprint(mol).GetNonzeroElements().items():
                fp_r2[idx % 2048] = count
            fp_r2_rows.append(fp_r2)

            fp_r3 = np.zeros(2048, dtype=np.float32)
            for idx, count in gen_r3.GetCountFingerprint(mol).GetNonzeroElements().items():
                fp_r3[idx % 2048] = count
            fp_r3_rows.append(fp_r3)

    if failed:
        print(f'  Warning: {failed} SMILES could not be parsed')

    # Combine into single DataFrame
    df_desc = pd.DataFrame(desc_rows, columns=rdkit_cols)
    df_fp_r2 = pd.DataFrame(fp_r2_rows, columns=[f'morgan_r2_{i}' for i in range(2048)])
    df_fp_r3 = pd.DataFrame(fp_r3_rows, columns=[f'morgan_r3_{i}' for i in range(2048)])

    result = pd.concat([df_desc, df_fp_r2, df_fp_r3], axis=1)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result

print('Computing features for xy_df (HOPV+SQL)...')
xy_features = compute_rdkit_features(xy_homo['smiles'].tolist(), RDKIT_COLS)
xy_features[TARGET] = xy_homo[TARGET].values
print(f'Done. Shape: {xy_features.shape}')
print(f'Total features: {xy_features.shape[1] - 1}')

## 3. Load & Compute Features for extra_data_200k

In [ ]:
extra = pd.read_csv('extra_data_200k.csv')
print(f'extra_data_200k shape: {extra.shape}')

print('Computing features for extra_data_200k...')
extra_features = compute_rdkit_features(extra['smiles'].tolist(), RDKIT_COLS)
extra_features[TARGET] = extra[TARGET].values
print(f'Done. Shape: {extra_features.shape}')

# Sla berekende features op als tussenbestand
extra_features.to_csv('extra_200k_features_homo.csv', index=False)
print('Features opgeslagen als extra_200k_features_homo.csv')


## 4. Combine Datasets

In [ ]:
FEATURE_COLS = [c for c in xy_features.columns if c != TARGET]

# Gebruik float32 om geheugen te besparen
xy_features[FEATURE_COLS] = xy_features[FEATURE_COLS].astype('float32')
extra_features[FEATURE_COLS] = extra_features[FEATURE_COLS].astype('float32')

combined = pd.concat(
    [xy_features, extra_features],
    ignore_index=True
)
combined = combined.dropna(subset=[TARGET]).reset_index(drop=True)

print(f'Combined training set: {combined.shape}')
print(f'  - From xy_df (HOPV+SQL): {len(xy_features)}')
print(f'  - From extra_200k: {len(extra_features)}')
print(f'HOMO range: {combined[TARGET].min():.3f} to {combined[TARGET].max():.3f}')

## 5. Prepare X and y

In [ ]:
X = combined[FEATURE_COLS].copy()
# Replace any remaining inf values with NaN
X = X.replace([np.inf, -np.inf], np.nan)
y = combined[TARGET].values

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'NaN in X: {X.isnull().sum().sum()}')
print(f'Inf in X: {np.isinf(X.select_dtypes(include=[float]).values).sum()}')

## 6. Preprocessing
### 6.1 Impute missing values

In [ ]:
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_COLS)
print(f'NaN after imputation: {X_imputed.isnull().sum().sum()}')

### 6.2 Remove near-zero variance features

In [ ]:
vt = VarianceThreshold(threshold=0.01)
X_var = vt.fit_transform(X_imputed)
kept_var = [FEATURE_COLS[i] for i in range(len(FEATURE_COLS)) if vt.get_support()[i]]
X_var = pd.DataFrame(X_var, columns=kept_var)
print(f'Features after variance filter: {len(kept_var)} (removed {len(FEATURE_COLS) - len(kept_var)})')

### 6.3 Remove highly correlated features (>0.95)
Only applied to RDKit descriptors — Morgan fingerprints are sparse and correlation filter would remove too many useful bits.

In [ ]:
# Split into RDKit descriptors and fingerprints
rdkit_kept = [c for c in kept_var if not c.startswith('morgan_')]
morgan_kept = [c for c in kept_var if c.startswith('morgan_')]

# Apply correlation filter only to RDKit descriptors
corr_matrix = X_var[rdkit_kept].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
rdkit_final = [c for c in rdkit_kept if c not in to_drop]

FINAL_FEATURES = rdkit_final + morgan_kept
X_filtered = X_var[FINAL_FEATURES]

print(f'RDKit features after correlation filter: {len(rdkit_final)} (removed {len(to_drop)})')
print(f'Morgan fingerprint features kept: {len(morgan_kept)}')
print(f'Total final features: {len(FINAL_FEATURES)}')

## 7. Baseline XGBoost — Cross-Validation

In [ ]:
X_train = X_filtered.copy()
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_train = X_train.values.astype(np.float64)

# Fix inf én extreme waarden
X_train[~np.isfinite(X_train)] = 0.0
X_train = np.clip(X_train, -1e10, 1e10)
y_train = y

print(f'X_train shape: {X_train.shape}')
print(f'Max waarde: {np.max(X_train):.4f}')
print(f'Min waarde: {np.min(X_train):.4f}')
print(f'Inf aanwezig: {np.isinf(X_train).any()}')
print('Data is clean!' if np.isfinite(X_train).all() else 'WARNING!')

## 8. Hyperparameter Tuning with Optuna

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.2, 0.5),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'device': 'cuda',
        'tree_method': 'hist',
        'missing': np.nan
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train,
                              cv=kf, scoring='neg_mean_absolute_error', n_jobs=1)
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest MAE: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
print(f"Max waarde in X_train: {np.max(X_train)}")
print(f"Min waarde in X_train: {np.min(X_train)}")
print(f"Inf aanwezig: {np.isinf(X_train).any()}")
print(f"NaN aanwezig: {np.isnan(X_train).any()}")

## 9. Train Final Model on All Data

In [ ]:
best_params = study.best_params
best_params['random_state'] = RANDOM_STATE
best_params['n_jobs'] = -1
best_params['tree_method'] = 'hist'

final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train, y_train)

train_preds = final_model.predict(X_train)
train_mae = mean_absolute_error(y_train, train_preds)
print(f'Train MAE (overfitting check): {train_mae:.4f}')
print(f'CV MAE (generalization):       {study.best_value:.4f}')

## 10. Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_train, train_preds, alpha=0.3, s=5, color='steelblue')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual HOMO (eV)')
axes[0].set_ylabel('Predicted HOMO (eV)')
axes[0].set_title(f'Predicted vs Actual\nTrain MAE: {train_mae:.4f}')

# Feature importance — top 20 RDKit descriptors only (fingerprints are too many to show)
importance = pd.Series(final_model.feature_importances_, index=FINAL_FEATURES)
rdkit_importance = importance[[c for c in FINAL_FEATURES if not c.startswith('morgan_')]]
rdkit_importance.nlargest(20).sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 20 RDKit Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('homo_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as homo_evaluation.png')

## 11. Save Model & Features

In [ ]:
joblib.dump(final_model, 'homo_model.joblib')
joblib.dump(FINAL_FEATURES, 'homo_features.joblib')
joblib.dump(RDKIT_COLS, 'homo_rdkit_cols.joblib')

print('Saved:')
print('  homo_model.joblib      — trained XGBoost model')
print('  homo_features.joblib   — final feature names (RDKit + Morgan)')
print('  homo_rdkit_cols.joblib — original RDKit descriptor names')
print(f'\nFinal CV MAE:      {study.best_value:.4f}')
print(f'Total features:    {len(FINAL_FEATURES)}')
print(f'  RDKit:           {len(rdkit_final)}')
print(f'  Morgan r2+r3:    {len(morgan_kept)}')
print(f'Training samples:  {len(X_train)}')